# Spotify Music Recommendation System
## Notebook 02 — Exploratory Data Analysis

**Purpose:** Understand the shape, distribution, and relationships in the data using simple, readable charts — one question per section.

## Table of Contents
1. [Setup & Imports](#1-setup--imports)
2. [Load Data](#2-load-data)
3. [Audio Feature Distributions](#3-audio-feature-distributions)
4. [Popularity Distribution](#4-popularity-distribution)
5. [Track Duration](#5-track-duration)
6. [Temporal Coverage](#6-temporal-coverage)
7. [Categorical Features](#7-categorical-features)
8. [Feature Correlations](#8-feature-correlations)
9. [How Features Changed Over Time](#9-how-features-changed-over-time)
10. [Genre Distribution (after fixing [] issue)](#10-genre-distribution)
11. [Top Artists](#11-top-artists)
12. [Key Findings](#12-key-findings)

## 1. Setup & Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100

DATA_DIR = Path('../data/raw')

## 2. Load Data

In [ ]:
data       = pd.read_csv(DATA_DIR / 'data.csv')
data_genre = pd.read_csv(DATA_DIR / 'data_w_genres.csv')
year_df    = pd.read_csv(DATA_DIR / 'data_by_year.csv')

print(f'Main dataset : {data.shape[0]:,} tracks, {data.shape[1]} columns')
print(f'Year range   : {data["year"].min()} – {data["year"].max()}')
data.head(3)

## 3. Audio Feature Distributions

All 7 features below are on a 0–1 scale. We look at their histogram shapes to understand how the 170,653 tracks are spread across each dimension.

**Skewed right (most tracks near 0):** `instrumentalness`, `speechiness`, `liveness`  
**Bell-shaped (spread across full range):** `valence`, `danceability`, `energy`, `acousticness`

In [ ]:
AUDIO = ['valence', 'acousticness', 'danceability', 'energy',
         'instrumentalness', 'liveness', 'speechiness']

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()
colors = sns.color_palette('husl', 7)

for i, feat in enumerate(AUDIO):
    axes[i].hist(data[feat], bins=50, color=colors[i], edgecolor='none')
    axes[i].axvline(data[feat].mean(), color='black', linewidth=1.5, linestyle='--',
                    label=f'mean = {data[feat].mean():.2f}')
    axes[i].set_title(feat, fontweight='bold')
    axes[i].set_xlabel('0 → 1')
    axes[i].legend(fontsize=8)

axes[-1].axis('off')
plt.suptitle('Audio Feature Distributions (170,653 tracks)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Skewness of each feature (higher = more right-skewed):')
print(data[AUDIO].skew().sort_values(ascending=False).round(2).to_string())

## 4. Popularity Distribution

Spotify popularity is a score from 0 to 100 based on how often a track has been played recently. There is a large spike at 0 — these are mostly very old or obscure tracks that nobody plays anymore.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(data['popularity'], bins=50, color='steelblue', edgecolor='none')
ax.axvline(data['popularity'].mean(),   color='red',    linewidth=1.5, linestyle='--',
           label=f'Mean = {data["popularity"].mean():.1f}')
ax.axvline(data['popularity'].median(), color='orange', linewidth=1.5, linestyle='--',
           label=f'Median = {data["popularity"].median():.0f}')
ax.set_title('Track Popularity Distribution', fontweight='bold')
ax.set_xlabel('Popularity (0 = unknown/old, 100 = very popular)')
ax.set_ylabel('Number of Tracks')
ax.legend()
plt.tight_layout()
plt.show()

zero_pop = (data['popularity'] == 0).sum()
print(f'Tracks with popularity = 0 : {zero_pop:,} ({zero_pop/len(data)*100:.1f}%)')
print(f'These are typically old or very obscure recordings.')

## 5. Track Duration

Most songs are 2–5 minutes long. A small number of classical/ambient pieces run much longer.

In [ ]:
data['duration_min'] = data['duration_ms'] / 60_000

fig, ax = plt.subplots(figsize=(10, 4))
# Clip to 15 min so the chart is readable
ax.hist(data['duration_min'].clip(upper=15), bins=60, color='mediumpurple', edgecolor='none')
ax.axvline(data['duration_min'].median(), color='red', linewidth=1.5, linestyle='--',
           label=f'Median = {data["duration_min"].median():.1f} min')
ax.set_title('Track Duration Distribution (clipped at 15 min for readability)', fontweight='bold')
ax.set_xlabel('Duration (minutes)')
ax.set_ylabel('Number of Tracks')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Median duration : {data["duration_min"].median():.2f} min')
print(f'Tracks < 1 min  : {(data["duration_min"] < 1).sum():,}  (interludes / intros — will filter)')
print(f'Tracks > 20 min : {(data["duration_min"] > 20).sum():,}  (classical / ambient pieces)')

## 6. Temporal Coverage

The dataset spans 100 years (1921–2020). Most tracks are from post-1990 because digital music catalogues grew rapidly after streaming arrived.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
decade = (data['year'] // 10 * 10)
decade_counts = decade.value_counts().sort_index()
ax.bar(decade_counts.index.astype(str), decade_counts.values,
       color=sns.color_palette('viridis', len(decade_counts)), width=7)
ax.set_title('Number of Tracks by Decade', fontweight='bold')
ax.set_xlabel('Decade')
ax.set_ylabel('Number of Tracks')
ax.tick_params(axis='x', rotation=45)
for x, y in zip(decade_counts.index.astype(str), decade_counts.values):
    ax.text(x, y + 200, f'{y:,}', ha='center', fontsize=7)
plt.tight_layout()
plt.show()

## 7. Categorical Features

Three categorical features: `explicit` (does the track have explicit lyrics), `mode` (major or minor key), and `key` (which of 12 musical keys).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Explicit
ec = data['explicit'].value_counts().sort_index()
axes[0].bar(['Clean (0)', 'Explicit (1)'], ec.values, color=['#2ecc71', '#e74c3c'], width=0.5)
axes[0].set_title('Explicit Tracks', fontweight='bold')
axes[0].set_ylabel('Tracks')
for i, v in enumerate(ec.values):
    axes[0].text(i, v + 500, f'{v:,}\n({v/len(data)*100:.1f}%)', ha='center', fontsize=9)

# Mode
mc = data['mode'].value_counts().sort_index()
axes[1].bar(['Minor (0)', 'Major (1)'], mc.values, color=['#3498db', '#9b59b6'], width=0.5)
axes[1].set_title('Track Mode (Major vs Minor)', fontweight='bold')
axes[1].set_ylabel('Tracks')
for i, v in enumerate(mc.values):
    axes[1].text(i, v + 500, f'{v:,}\n({v/len(data)*100:.1f}%)', ha='center', fontsize=9)

# Key
key_labels = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
kc = data['key'].value_counts().sort_index()
axes[2].bar(key_labels, kc.values, color=sns.color_palette('husl', 12))
axes[2].set_title('Musical Key Distribution', fontweight='bold')
axes[2].set_xlabel('Key')
axes[2].set_ylabel('Tracks')
axes[2].tick_params(axis='x', rotation=45)

plt.suptitle('Categorical Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Feature Correlations

This heatmap shows how audio features relate to each other. Strong relationships:
- **Energy ↔ Loudness**: +0.76 (louder = more energetic)
- **Acousticness ↔ Energy**: −0.72 (acoustic tracks tend to be quiet and calm)
- **Popularity** has only weak links to audio features — sound alone does not predict popularity.

In [ ]:
NUM_COLS = ['valence', 'acousticness', 'danceability', 'energy',
            'instrumentalness', 'liveness', 'loudness', 'speechiness',
            'tempo', 'popularity']

corr = data[NUM_COLS].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))  # hide upper triangle
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.4, ax=ax,
            annot_kws={'size': 8}, cbar_kws={'shrink': 0.7})
ax.set_title('Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. How Features Changed Over Time

Using `data_by_year.csv` (mean values per year, 1921–2020).

Key trends:
- **Acousticness** has fallen steadily since the 1960s as electric recording took over
- **Energy and Loudness** have increased — the famous 'loudness war'
- **Danceability** has slowly increased over the decades

In [ ]:
# Drop constant mode column before plotting
TREND_FEATS = ['acousticness', 'energy', 'danceability', 'valence', 'loudness']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
colors = sns.color_palette('tab10', len(TREND_FEATS))

for i, feat in enumerate(TREND_FEATS):
    axes[i].plot(year_df['year'], year_df[feat], color=colors[i], linewidth=2)
    axes[i].set_title(feat, fontweight='bold')
    axes[i].set_xlabel('Year')
    axes[i].set_ylabel('Mean value')
    axes[i].axvspan(1990, 2020, alpha=0.07, color='gray', label='Post-1990')
    axes[i].legend(fontsize=8)

axes[-1].axis('off')
plt.suptitle('How Audio Features Changed Over Time (yearly averages)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 10. Genre Distribution

> **Important:** The `genres` column in `data_w_genres.csv` stores values as Python list strings. Many rows contain `'[]'` which looks non-null but means **no genre**. We must filter these out before any genre analysis.

- 9,857 artists (34.3%) have `genres = '[]'` — empty, not a real genre
- Only the remaining 18,823 artists have actual genre information

In [ ]:
def parse_genre_list(s):
    """Parse '[]' or "['pop', 'rock']" into a Python list."""
    try:
        result = ast.literal_eval(s)
        return result if isinstance(result, list) else []
    except Exception:
        return []

data_genre['genres_parsed'] = data_genre['genres'].apply(parse_genre_list)

# Show the [] problem clearly
empty_genre_count = data_genre['genres_parsed'].apply(len).eq(0).sum()
print(f'Rows with genres = []  : {empty_genre_count:,}  ← hidden empty values')
print(f'Rows with real genres  : {len(data_genre) - empty_genre_count:,}')
print()

# Explode into individual genre tags — ONLY from artists with real genres
genre_series = data_genre['genres_parsed'].explode()
genre_series = genre_series[genre_series.str.strip() != '']  # remove empty strings
top_genres = genre_series.value_counts().head(20)

fig, ax = plt.subplots(figsize=(10, 7))
top_genres.plot(kind='barh', ax=ax, color=sns.color_palette('viridis', len(top_genres)))
ax.invert_yaxis()
ax.set_title('Top 20 Genres by Artist Count\n([] empty genres excluded)', fontweight='bold')
ax.set_xlabel('Number of Artists')
plt.tight_layout()
plt.show()

print(f'Total unique genre tags: {genre_series.nunique()}')

## 11. Top Artists

In [ ]:
# artists in data.csv are list strings — parse them first
def parse_artist_list(s):
    try:
        result = ast.literal_eval(s)
        return result if isinstance(result, list) else [str(result)]
    except Exception:
        return [str(s)]

all_artists = data['artists'].apply(parse_artist_list).explode()
top_artists = all_artists.value_counts().head(20)

fig, ax = plt.subplots(figsize=(10, 7))
top_artists.plot(kind='barh', ax=ax, color=sns.color_palette('husl', len(top_artists)))
ax.invert_yaxis()
ax.set_title('Top 20 Artists by Number of Tracks', fontweight='bold')
ax.set_xlabel('Number of Tracks')
plt.tight_layout()
plt.show()

## 12. Key Findings

These findings directly guide how we clean and use the data:

| Finding | What to do |
|---|---|
| `instrumentalness`, `speechiness`, `liveness` are right-skewed | Apply log transform before modelling |
| 34% of `genres` values are hidden-empty `[]` | Filter `[]` before any genre analysis or join |
| `artists` in `data.csv` are list strings | Parse with `ast.literal_eval()` in preprocessing |
| `popularity = 0` for 15%+ of tracks | Flag these — they are obscure, not missing |
| `mode` in `data_by_year` is always 1 | Drop this column |
| `release_date` has mixed formats | Use `year` column instead |
| Tracks < 1 min exist (interludes) | Filter out in cleaning |
| Energy ↔ Loudness strongly correlated (+0.76) | Consider this when building feature vectors |